*Alunos: Bruno Calabria Cortez Navas // Eduardo Brioso Luceiro*

In [1]:
import pandas as pd
import numpy as np
from scipy import stats
import statsmodels.api as sm
from statsmodels.formula.api import ols

In [2]:
# Leitura e exibição do banco de dados

file = pd.read_csv("./VG_Sales.csv")
file = file.dropna() # Remoção de lihas com dados vazios
file

,Name,Console,Year_of_Release,Genre,Publisher,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales,Critic_Score,Critic_Count,User_Score,User_Count,Developer,Rating
0,Wii Sports,Wii,2006.0,Sports,Nintendo,41.36,28.96,3.77,8.45,82.53,76.0,51.0,8,322.0,Nintendo,E
2,Mario Kart Wii,Wii,2008.0,Racing,Nintendo,15.68,12.76,3.79,3.29,35.52,82.0,73.0,8.3,709.0,Nintendo,E
3,Wii Sports Resort,Wii,2009.0,Sports,Nintendo,15.61,10.93,3.28,2.95,32.77,80.0,73.0,8,192.0,Nintendo,E
6,New Super Mario Bros.,DS,2006.0,Platform,Nintendo,11.28,9.14,6.50,2.88,29.80,89.0,65.0,8.5,431.0,Nintendo,E
7,Wii Play,Wii,2006.0,Misc,Nintendo,13.96,9.18,2.93,2.84,28.92,58.0,41.0,6.6,129.0,Nintendo,E
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16667,E.T. The Extra-Terrestrial,GBA,2001.0,Action,NewKidCo,0.01,0.00,0.00,0.00,0.01,46.0,4.0,2.4,21.0,Fluid Studios,E
16677,Mortal Kombat: Deadly Alliance,GBA,2002.0,Fighting,Midway Games,0.01,0.00,0.00,0.00,0.01,81.0,12.0,8.8,9.0,Criterion Games,M
16696,Metal Gear Solid V: Ground Zeroes,PC,2014.0,Action,Konami Digital Entertainment,0.00,0.01,0.00,0.00,0.01,80.0,20.0,7.6,412.0,Kojima Productions,M
16700,Breach,PC,2011.0,Shooter,Destineer,0.01,0.00,0.00,0.00,0.01,61.0,12.0,5.8,43.0,Atomic Games,T


## Estimação de parâmetros
Nesta seção, calcularemos a média geral de notas de usuário e a média geral de notas de empresas a fim de verificar se são similares.

In [3]:
# Transforma o campo User_Score do CSV em dados numéricos
file['User_Score'] = pd.to_numeric(file['User_Score'], errors='coerce')
file.dropna(subset=['User_Score'], inplace=True)

# Média e desvio padrão das notas de usuários
mean_userScore = file['User_Score'].mean()
std_userScore = file['User_Score'].std()
n_userScore = file['User_Score'].count()

print(f"Nota média de usuários: {mean_userScore:.2f}")
print(f"Desvio padrão de notas de usuários: {std_userScore:.2f}")

# Utilizando um intervalo de confiança de 95%
# Como o desvio padrão da população é desconhecido, podemos usar distribuição t para calcular a média neste intervalo de confiança
confianca = 0.95
liberdade = n_userScore - 1
erro = std_userScore / np.sqrt(n_userScore)

intervaloConfianca = stats.t.interval(confianca, liberdade,
                                       loc=mean_userScore,
                                       scale=erro)

print(f"Intervalo de confiança de 95% para nota de usuários: ({intervaloConfianca[0]:.2f}, {intervaloConfianca[1]:.2f})")

Nota média de usuários: 7.19
Desvio padrão de notas de usuários: 1.44
Intervalo de confiança de 95% para nota de usuários: (7.15, 7.22)


In [4]:
# Transforma o campo Critic_Score do CSV em dados numéricos
file['Critic_Score'] = pd.to_numeric(file['Critic_Score'], errors='coerce')
file.dropna(subset=['Critic_Score'], inplace=True)

# As notas da crítica podem ser dadas de 0 a 100, enquanto as de usuário vão de 0 a 10. Por isso, vamos ajustar os dados na hora de mostrá-los.

# Média e desvio padrão das notas de crítica
mean_criticScore = file['Critic_Score'].mean()
std_criticScore = file['Critic_Score'].std()
n_criticScore = file['Critic_Score'].count()

mean_criticScoreN = mean_criticScore/10
std_criticScoreN = std_criticScore/10
print(f"Nota média da crítica: {mean_criticScoreN:.2f}")
print(f"Desvio padrão de notas de crítica: {std_criticScoreN:.2f}")

# Utilizando um intervalo de confiança de 95%
# Como o desvio padrão da população é desconhecido, podemos usar distribuição t para calcular a média neste intervalo de confiança
confianca = 0.95
liberdade = n_criticScore - 1
erro = std_criticScore / np.sqrt(n_criticScore)

intervaloConfianca = stats.t.interval(confianca, liberdade,
                                       loc=mean_criticScore,
                                       scale=erro)

intervaloConfianca0N = intervaloConfianca[0]/10
intervaloConfianca1N = intervaloConfianca[1]/10
print(f"Intervalo de confiança de 95% para nota da crítica: ({intervaloConfianca0N:.2f}, {intervaloConfianca1N:.2f})")

Nota média da crítica: 7.03
Desvio padrão de notas de crítica: 1.39
Intervalo de confiança de 95% para nota da crítica: (6.99, 7.06)


## Teste de hipóteses
A seguir, testaremos duas hipóteses:
1. A empresa desenvolvedora tem grande influência no desempenho comercial do jogo
2. As notas de usuários e da crítica não impactam significativamente no sucesso comercial do jogo.
Ambas relações foram estudadas na parte 1 do trabalho utilizando outros métodos. Aqui, verificaremos se o resultado é o mesmo ao utilizarmos outras metodologias.

## Regressão linear
Aqui, verificaremos se é possível prever as vendas globais com base em vendas de determinada região.

In [5]:
# Variável independente: Vendas na América do Norte
# Variável dependente: Vendas Globais
X = file['NA_Sales']
y = file['Global_Sales']

# Adiciona uma constante à variável dependente 
X = sm.add_constant(X)

# Cria e aplica o modelo OLS (Ordinary Leas Squares ou Modelo dos Mínimos Quadrados)
model = sm.OLS(y, X)
resultados = model.fit()

print(f"\n--- Regressão Linear Vendas América do Norte vs Vendas Globais ---")
print(resultados.summary())

# Interpretation
print(f"\nEquation: Global_Sales = {resultados.params['const']:.2f} + {resultados.params['NA_Sales']:.2f} * NA_Sales")
print(f"R-squared: {resultados.rsquared:.3f}")
print(f"P-value for NA_Sales coefficient: {resultados.pvalues['NA_Sales']:.3e}")


--- Regressão Linear Vendas América do Norte vs Vendas Globais ---
                            OLS Regression Results                            
Dep. Variable:           Global_Sales   R-squared:                       0.914
Model:                            OLS   Adj. R-squared:                  0.914
Method:                 Least Squares   F-statistic:                 7.209e+04
Date:                Wed, 11 Jun 2025   Prob (F-statistic):               0.00
Time:                        17:38:38   Log-Likelihood:                -5934.5
No. Observations:                6825   AIC:                         1.187e+04
Df Residuals:                    6823   BIC:                         1.189e+04
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------

## ANOVA
Com este método, faremos as seguintes comparações:
1. Quantidade de vendas entre empresas
2. Quantidade de vendas entre consoles
3. Quantidade de vendas entre gêneros

In [6]:
empresasComparacao = ['Nintendo', 'Microsoft Game Studios', 'Take-Two Interactive', 'Activision', 'Sony Computer Entertainment', 'Ubisoft', 'Electronic Arts']
dfFiltrado = file[file['Publisher'].isin(empresasComparacao)]

vendasEmpresas = [dfFiltrado[dfFiltrado['Publisher']==empresa]['Global_Sales'] for empresa in empresasComparacao]

f, p = stats.f_oneway(*vendasEmpresas)

print(f"---ANOVA Vendas Globais por Empresa---")
print(f"Estatística F: {f:.3f}")
print(f"p: {p:.3e}")


---ANOVA Vendas Globais por Empresa---
Estatística F: 24.110
p: 5.293e-28


Considerando nível de significância 0.05, como $p\lt\lt0.05$, percebe-se que a empresa que publica o jogo tem impacto significativo no seu desempenho comercial. Chegamos à essa conclusão na primeira parte do trabalho e podemos ver que, mesmo utilizando outros métodos, ela se mantém.

In [7]:
consolesComparacao = ['Wii', 'NES', 'GB', 'DS', 'X360', 'PS', 'PS2', 'PS3', 'PS4', 'SNES', 'N64', '3DS', 'XB', 'XONE']
df = file[file['Console'].isin(consolesComparacao)]

vendasConsoles = [df[df['Console']==console]['Global_Sales'] for console in consolesComparacao]

f, p = stats.f_oneway(*vendasConsoles)

print(f"---ANOVA Vendas Globais por Console---")
print(f"Estatística F: {f:.3f}")
print(f"p: {p:.3e}")


---ANOVA Vendas Globais por Console---
Estatística F: nan
p: nan


/tmp/ipykernel_2633/1331555403.py:6: SmallSampleWarning: One or more sample arguments is too small; all returned values will be NaN. See documentation for sample size requirements.
  f, p = stats.f_oneway(*vendasConsoles)


In [8]:
generoComparacao = ['Sports', 'Action', 'Role-Playing', 'Racing', 'Platform', 'Puzzle']
dfFiltrado = file[file['Genre'].isin(generoComparacao)]

vendasGenero = [dfFiltrado[dfFiltrado['Genre']==genero]['Global_Sales'] for genero in generoComparacao]

f, p = stats.f_oneway(*vendasGenero)

print(f"---ANOVA Vendas globais por gênero---")
print(f"Estatística F: {f:.3f}")
print(f"p: {p:.3e}")


---ANOVA Vendas globais por gênero---
Estatística F: 1.290
p: 2.650e-01


Considerando nível de significância 0.05, vê-se que, como $p>0.05$, então o gênero do jogo não tem impacto significativo no seu número de vendas.

## Qui-quadrado
Por fim, tentaremos responder as seguintes perguntas:
1. O gênero é independente da classificação etária do jogo?
2. Qual a relação entre o período de tempo e o número de vendas?

In [9]:
tabela = pd.crosstab(file['Genre'], file['Rating'])

chi2, p, liberdade, esperado = stats.chi2_contingency(tabela)

print(f"\n--- Teste de independência Qui-Quadrado (Gênero vs Classificação etária) ---")
print(f"Estatística qui-quadrado: {chi2:.3f}")
print(f"P: {p:.3e}")
print(f"Graus de liberdade: {liberdade}")


--- Teste de independência Qui-Quadrado (Gênero vs Classificação etária) ---
Estatística qui-quadrado: 3993.758
P-value: 0.000e+00
Degrees of Freedom: 66


In [11]:
tabela = pd.crosstab(file['Year_of_Release'], file['Global_Sales'])

chi2, p, liberdade, esperado = stats.chi2_contingency(tabela)

print(f"\n--- Teste de independência Qui-Quadrado (Ano de lançamento vs Vendas Globais) ---")
print(f"Estatística qui-quadrado: {chi2:.3f}")
print(f"P: {p:.3e}")
print(f"Graus de liberdade: {liberdade}")


--- Teste de independência Qui-Quadrado (Ano de lançamento vs Vendas Globais) ---
Estatística qui-quadrado: 17814.707
P-value: 7.518e-170
Degrees of Freedom: 12840


Sendo o nível de significância 0.05, nota-se que $p<0.05$. Por isso, as variáveis possuem uma associação estatisticamente significativa.